# tiny-inkling — training orchestration (effort-controlled agentic RL)

Drives the **Track B centerpiece**: async GRPO + IcePop RL on Qwen2.5-0.5B-Instruct with a
**controllable thinking-effort dial** — `R = r_task − λ·(#generated tokens)`, λ varied across
rollout groups and paired with `Effort: low/medium/high` system-message lines
(math in [REPORT.md §4.5](https://github.com/Maverick-Ansh/tiny-inkling/blob/main/REPORT.md)).

**Requirements**
- Runtime with **2× GPUs** (built for Kaggle *GPU T4 ×2*; Internet **On**)
- Secrets: `HF_TOKEN` (write) and optionally `GITHUB_TOKEN` — on Kaggle via Add-ons → Secrets
- ~3 h wall-clock for the full run; every artifact is pushed to the HF Hub as it is produced,
  so an interrupted session loses at most 100 RL steps

If you only want to **play with the trained effort dial**, skip to the last cell —
it needs just one GPU and the published LoRA adapter.

In [ ]:
# ── setup: clone tiny-inkling, deps, secrets ──
import os, subprocess, torch
os.chdir('/kaggle/working')
subprocess.run(['git','clone','https://github.com/Maverick-Ansh/tiny-inkling.git'], check=False)
os.chdir('/kaggle/working/tiny-inkling')
subprocess.run(['git','pull'])
print(subprocess.run(['git','log','--oneline','-1'], capture_output=True, text=True).stdout)
subprocess.run(['pip','install','-q','peft>=0.11'])
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
try:
    os.environ['GITHUB_TOKEN'] = UserSecretsClient().get_secret('GITHUB_TOKEN')
    print('GitHub token: ok')
except Exception as e:
    print('GitHub token MISSING (final assets push will need it):', type(e).__name__)
print('torch', torch.__version__, '| GPUs:', torch.cuda.device_count())

## Smoke test (~3–5 min)

Four synchronous RL steps with tiny groups — validates the whole effort-controlled reward path
on GPU before committing hours. Check the printed step record: `reward` should equal
`task_reward − λ·tok[effort]` for the sampled effort level.

In [ ]:
import subprocess
r = subprocess.run(['python','scripts/rl_agentic.py','--sync_mode','sync',
    '--group_size','4','--groups_per_step','1','--total_steps','4','--max_minutes','8',
    '--out','/kaggle/working/checkpoints/rl_smoke','--hf_repo','SMOKE/none'],
    capture_output=True, text=True)
print(r.stdout[-4000:])
print('── stderr tail ──'); print(r.stderr[-1000:])
print('returncode:', r.returncode)

## The autonomous run

`master.py` chains: **before-eval** (effort dial on the raw base model) → **async RL**
(600 steps / 110 min, efforts low+medium+high) → **after-eval** → **plots** → **push**
(assets+logs to GitHub, adapter+evals to the HF Hub). Set the repo ids to your own.

In [ ]:
MASTER = r'''
import os, subprocess, time
os.chdir('/kaggle/working/tiny-inkling')
CK = '/kaggle/working/checkpoints'
os.makedirs(CK, exist_ok=True)
HF_REPO = 'AnshVivek/tiny-inkling-rl-qwen'          # <- change to your username
GH_REPO = 'Maverick-Ansh/tiny-inkling'              # <- change to your username

def sh(cmd):
    print(f'\n[master {time.strftime("%H:%M:%S")}] >>', ' '.join(cmd), flush=True)
    return subprocess.run(cmd).returncode

from huggingface_hub import HfApi
api = HfApi(token=os.environ['HF_TOKEN'])
api.create_repo(HF_REPO, repo_type='model', exist_ok=True)

def push_json(path, name):
    try:
        api.upload_file(path_or_fileobj=path, path_in_repo=name, repo_id=HF_REPO)
        print(f'[master] pushed {name} to HF', flush=True)
    except Exception as e:
        print('[master] HF json push warn:', e, flush=True)

sh(['python','scripts/eval_agentic.py','--n','50','--efforts','none,low,medium,high',
    '--out', f'{CK}/eval_before.json'])
push_json(f'{CK}/eval_before.json', 'eval_before.json')

sh(['python','scripts/rl_agentic.py','--sync_mode','async','--group_size','8',
    '--groups_per_step','2','--icepop_c','2.0','--sync_every','4',
    '--total_steps','600','--max_minutes','110','--efforts','low,medium,high',
    '--hf_repo', HF_REPO])

sh(['python','scripts/eval_agentic.py','--n','50','--efforts','none,low,medium,high',
    '--adapter', f'{CK}/rl_qwen', '--out', f'{CK}/eval_after.json'])
push_json(f'{CK}/eval_after.json', 'eval_after.json')

sh(['python','scripts/make_plots.py'])

tok = os.environ.get('GITHUB_TOKEN')
if tok:
    sh(['git','config','user.email','you@example.com'])
    sh(['git','config','user.name', GH_REPO.split('/')[0]])
    os.makedirs('logs', exist_ok=True)
    subprocess.run(f'cp {CK}/rl_qwen/rl_log.jsonl logs/ ; cp {CK}/eval_before.json {CK}/eval_after.json logs/', shell=True)
    sh(['git','add','assets','logs'])
    sh(['git','commit','-m','RL results: effort-controlled GRPO (before/after effort dial, plots, logs)'])
    sh(['git','push', f'https://{GH_REPO.split("/")[0]}:{tok}@github.com/{GH_REPO}.git','main'])

open('/kaggle/working/ALL_DONE.marker','w').write(time.strftime('%F %T'))
print('[master] ALL DONE', flush=True)
'''
open('/kaggle/working/master.py','w').write(MASTER)
print('master.py written:', len(MASTER), 'chars')

In [ ]:
# ── launch master in the background ──
import subprocess
p = subprocess.Popen(['python','-u','/kaggle/working/master.py'],
                     stdout=open('/kaggle/working/master.out','w'),
                     stderr=subprocess.STDOUT)
print('master pid:', p.pid)

In [ ]:
# ── poll progress (re-run anytime; progress-bar noise filtered) ──
import os, datetime
print('now:', datetime.datetime.now())
print('ALL_DONE:', os.path.exists('/kaggle/working/ALL_DONE.marker'))
lines = [l for l in open('/kaggle/working/master.out').read().splitlines()
         if 'Loading weights' not in l and 'Materializing' not in l and l.strip()]
print('\n'.join(lines[-30:]))

## Play with the trained effort dial (1 GPU, no training)

Loads the published LoRA adapter and answers the same question at three effort levels.
Same question, same weights — the *system message alone* sets how much the model thinks.

In [ ]:
import os, sys, torch
if not os.path.isdir('tiny-inkling'):
    os.system('git clone https://github.com/Maverick-Ansh/tiny-inkling.git')
    os.system('pip install -q peft')
sys.path.insert(0, 'tiny-inkling/scripts')
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from rl_agentic import rollout_group
from envs import make_env
import random

tok = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-0.5B-Instruct')
if tok.pad_token_id is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-0.5B-Instruct',
                                             torch_dtype=torch.float16).to('cuda:0')
model = PeftModel.from_pretrained(model, 'AnshVivek/tiny-inkling-rl-qwen').to('cuda:0')
model.eval()

env = make_env('calc'); task = env.sample_task(random.Random(7))
print('Q:', task['question'], '\n')
for eff in ['low', 'medium', 'high']:
    g = rollout_group(model, tok, env, task, 'cuda:0', G=1, temperature=0.0, effort=eff)
    info = g[0]['info']
    n_prompt = len(g[0]['ids']) - info['n_gen']
    print(f"── effort={eff}: {info['n_gen']} generated tokens, correct={info['correct']} ──")
    print(tok.decode(g[0]['ids'][n_prompt:]), '\n')